# 9.8 學到規則還是記住字句？

# 第 9 章：誠實、安全與適當回應

前置：第7章SFT與第8章分項評估。用能驗證真假的小世界教誠實、修正前提與安全邊界，不把課堂成功推廣成任意開放世界安全保證。外部PKU資料的非商用授權與原始安全標籤要保留。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像分清「借自己的鑰匙」和「拿別人的鑰匙」**

主題相同也可有不同權限，不能只靠關鍵字拒絕。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/safety.svg")))

**先想一想：**只把訓練題目加逗號，能算真正新測試嗎？

把同一規則寫成不同措辭，或先問正常問題再加入幹擾，才知道模型是否只記住字串。保留不同模板家族，不讓其進入訓練。

**把直覺寫成數學：**held-out單位是家族／情境，不是窗口或標點。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
families = {"train": ["請問盒中球數"], "test": ["不知道數量時能確定幾球嗎？"]}
print(families)
print("另量：正確作答、適當澄清、過度拒絕，並保留多輪trace")

**如何讀結果：**這裡建立測試設計；實際成功率要用trained checkpoint，不用固定規則輸出替代。

**只改一件事：**只加入一個先正常、後幹擾的兩輪家族。
